# 0 - Téléchargement des données

Ce notebook illustre les différentes fonctionnalités des clients du répertoire pour télécharger des données via l'API SDMX. Chaque client permet de télécharger des données depuis une source distincte (Eurostat etc ...)

## Table des matières

0. [Importation des modules](#section-0)
1. [Téléchargement des données d'Eurostat](#section-1)
    - 1.1 [Énumération des dataflows](#section-1.1)
    - 1.2 [Description de la structure d'un dataflow](#section-1.2)
    - 1.3 [Requête basique avec noms de dimensions](#section-1.3)
    - 1.4 [Filtres temporels](#section-1.4)
    - 1.5 [Formats de réponse](#section-1.5)
    - 1.6 [Séparation des requêtes avec split dimensions](#section-1.6)
    - 1.7 [Utilisation de EurostatQueryRequest](#section-1.7)
    - 1.8 [Datasets Comext (DS-*)](#section-1.8)
    - 1.9 [Utilisation en tant que context manager](#section-1.9)
    - 1.10 [Gestion des erreurs](#section-1.10)
    - 1.11 [Memento et conseils de performance](#section-1.11)

## 0 - Importation des modules <a id="section-0"></a>

Importation des modules nécessaires et configuration de l'environnement.

In [ ]:
# Rechargement automatique des modules
%load_ext autoreload
%autoreload 2

# Modules de base
import sys
import pandas as pd
from pathlib import Path

# Ajout du répertoire parent au path
sys.path.append('..')

# Modules du package
from macroforecast.datasets.sources import EurostatClient, EurostatQueryRequest, EurostatResponseFormat

## 1 - Téléchargement des données d'Eurostat <a id="section-1"></a>

In [ ]:
# Initialisation du client
client = EurostatClient()

### 1.1 - Énumération des dataflows <a id="section-1.1"></a>

La méthode `list_all_dataflows()` permet d'énumérer tous les dataflows disponibles auprès d'Eurostat.

In [ ]:
# Récupération de tous les dataflows
dataflows = client.list_all_dataflows()

# Affichage
print(f"Nombre total de dataflows: {len(dataflows)}")
print("\nPremiers dataflows:")
dataflows.head(10)

### 1.2 - Description de la structure d'un dataflow <a id="section-1.2"></a>

La méthode `get_structure()` permet d'extraire les dimensions d'un dataflow. Contrairement au client OCDE, l'agence est toujours `ESTAT` et n'est pas un paramètre.

In [ ]:
# Extraction de la structure du dataflow namq_10_gdp (comptes nationaux trimestriels)
structure = client.get_structure(
    dataflow="namq_10_gdp",
    version="*"
)

# Affichage
print(f"Agency: {structure.agency}")
print(f"Dataflow: {structure.dataflow}")
print(f"Nombre de dimensions: {structure.num_dimensions}")
print("\nDimensions:")
for dim in structure.dimensions:
    print(f"  Position {dim.position}: {dim.name} - {dim.description}")

### 1.3 - Requête basique avec noms de dimensions <a id="section-1.3"></a>

La méthode `get_data()` permet de requêter les données d'un dataflow en appliquant des filtres sur les dimensions désignées par leur nom. Le paramètre `dimensions` accepte aussi bien une valeur unique (`str`) qu'une liste de valeurs (`List[str]`).

In [ ]:
# Requête sur le dataflow namq_10_gdp (PIB trimestriel)
df = client.get_data(
    dataflow="namq_10_gdp",
    dimensions={
        "GEO": ["FR", "DE"],  # France et Allemagne
        "FREQ": "Q",           # Fréquence trimestrielle
        "NA_ITEM": "B1GQ",     # PIB aux prix du marché
        "UNIT": "CP_MEUR",     # Millions d'euros courants
    },
    start_period="2020"
)

# Affichage
print(f"Nombre de lignes: {len(df)}")
print(f"Colonnes: {list(df.columns)}")
print(f"Pays: {sorted(df['GEO'].unique())}")
print(f"Période: {df['TIME_PERIOD'].min()} - {df['TIME_PERIOD'].max()}")
df.head()

### 1.4 - Filtres temporels <a id="section-1.4"></a>

La méthode `get_data()` propose plusieurs paramètres pour filtrer les observations dans le temps :
- `start_period` / `end_period` : borne inférieure et supérieure de la période (format SDMX, ex. `"2020-Q1"`, `"2024"`)
- `last_n_observations` : N dernières observations seulement
- `first_n_observations` : N premières observations seulement

In [ ]:
# Filtre par période de début et de fin
df_period = client.get_data(
    dataflow="namq_10_gdp",
    dimensions={
        "GEO": "FR",
        "FREQ": "Q",
        "NA_ITEM": "B1GQ",
        "UNIT": "CP_MEUR",
    },
    start_period="2022-Q1",
    end_period="2024-Q4"
)

print("--- Filtre start_period / end_period ---")
print(f"Nombre de lignes: {len(df_period)}")
print(f"Période: {df_period['TIME_PERIOD'].min()} - {df_period['TIME_PERIOD'].max()}")

In [ ]:
# Récupération des N dernières observations uniquement
df_last = client.get_data(
    dataflow="namq_10_gdp",
    dimensions={
        "GEO": "FR",
        "FREQ": "Q",
        "NA_ITEM": "B1GQ",
        "UNIT": "CP_MEUR",
    },
    last_n_observations=4  # Quatre derniers trimestres
)

print("--- last_n_observations=4 ---")
print(f"Nombre de lignes: {len(df_last)}")
df_last

### 1.5 - Formats de réponse <a id="section-1.5"></a>

L'API SDMX 3.0 d'Eurostat supporte plusieurs formats de réponse, encapsulés dans l'énumération `EurostatResponseFormat` :
- **CSV** (défaut) : SDMX-CSV 3.0, format tidy en colonnes
- **TSV** : format héritage Eurostat en colonnes larges avec flags
- **JSON** : JSON-stat 2.0
- **XML** : SDMX-ML 3.0 (données structurées)

In [ ]:
# Affichage des formats disponibles
print("Formats de réponse disponibles:")
for fmt in EurostatResponseFormat:
    print(f"  - {fmt.name}: '{fmt.value}'")

In [ ]:
# Paramètres communs à la requête de comparaison
common_kwargs = dict(
    dataflow="une_rt_m",  # Taux de chômage mensuel
    dimensions={
        "GEO": "FR",
        "FREQ": "M",
        "AGE": "TOTAL",
        "SEX": "T",
        "UNIT": "PC_ACT",
    },
    start_period="2024-01",
    end_period="2024-06",
)

# Requête au format CSV (défaut)
df_csv = client.get_data(**common_kwargs, format=EurostatResponseFormat.CSV)

# Requête au format JSON
df_json = client.get_data(**common_kwargs, format=EurostatResponseFormat.JSON)

# Comparaison des résultats
print("--- Format CSV ---")
print(f"Colonnes: {list(df_csv.columns)}")
print(df_csv[["GEO", "TIME_PERIOD", "OBS_VALUE"]].head(3).to_string(index=False))

print("\n--- Format JSON ---")
print(f"Colonnes: {list(df_json.columns)}")
print(df_json.head(3).to_string(index=False))

### 1.6 - Séparation des requêtes avec split dimensions <a id="section-1.6"></a>

Le paramètre `split_dimensions` de la méthode `get_data()` permet d'effectuer une requête distincte pour chaque combinaison de valeurs des dimensions spécifiées. Cela permet de gérer les requêtes volumineuses ou de contourner les limitations de l'API.

In [ ]:
# Requête avec split_dimensions
# Au lieu d'une seule requête pour 6 pays, on génère 6 requêtes séparées
df = client.get_data(
    dataflow="namq_10_gdp",
    dimensions={
        "GEO": ["FR", "DE", "IT", "ES", "NL", "BE"],
        "FREQ": "Q",
        "NA_ITEM": "B1GQ",
        "UNIT": "CP_MEUR",
    },
    start_period="2022",
    split_dimensions=["GEO"]  # Génère 6 requêtes séparées, une par pays
)

# Affichage
print(f"Nombre de lignes: {len(df)}")
print(f"Pays: {sorted(df['GEO'].unique())}")
print("\nNombre d'observations par pays:")
print(df.groupby('GEO').size())

### 1.7 - Utilisation de EurostatQueryRequest <a id="section-1.7"></a>

L'objet `EurostatQueryRequest` encapsule tous les paramètres d'une requête et peut être passé à `execute_query()`. Il facilite la manipulation programmatique des requêtes (stockage, modification, sérialisation).

In [ ]:
# Création d'une EurostatQueryRequest
query = EurostatQueryRequest(
    dataflow="namq_10_gdp",
    version="*",
    dimensions={
        "GEO": ["FR", "DE"],
        "FREQ": "Q",
        "NA_ITEM": "B1GQ",
        "UNIT": "CP_MEUR",
    },
    start_period="2020"
)

# Affichage des métadonnées de la requête
print(f"Dataflow key: {query.get_dataflow_key()}")
print(f"Dimensions: {query.dimensions}")
print(f"Format: {query.format}")

# Exécution de la requête
df = client.execute_query(query)

# Affichage
print(f"\nNombre de lignes: {len(df)}")
print(f"Période: {df['TIME_PERIOD'].min()} - {df['TIME_PERIOD'].max()}")
df.head()

In [ ]:
# Modification d'une QueryRequest existante (ex. ajout d'un pays)
query_extended = EurostatQueryRequest(
    **{
        **query.to_dict(),  # Reprise des paramètres existants
        "dimensions": {
            **query.dimensions,
            "GEO": ["FR", "DE", "IT"],  # Ajout de l'Italie
        },
        "last_n_observations": 8,       # Restriction aux 8 dernières observations
    }
)

print(f"Pays dans la requête étendue: {query_extended.dimensions['GEO']}")
print(f"Dernières observations: {query_extended.last_n_observations}")

# Exécution
df_extended = client.execute_query(query_extended)
print(f"\nNombre de lignes: {len(df_extended)}")

### 1.8 - Datasets Comext (DS-*) <a id="section-1.8"></a>

Les datasets dont l'identifiant commence par `DS-` proviennent de la base **Comext** (données de commerce extérieur). Le client détecte automatiquement ces datasets et bascule vers l'API Comext dédiée (`ec.europa.eu/eurostat/api/comext/...`). Aucune configuration supplémentaire n'est requise.

In [ ]:
# Vérification de la détection automatique Comext
comext_dataflow = "DS-045409"
standard_dataflow = "namq_10_gdp"

print("Détection automatique des datasets Comext:")
print(f"  '{comext_dataflow}' → Comext: {client._is_comext_dataset(comext_dataflow)}")
print(f"  '{standard_dataflow}' → Comext: {client._is_comext_dataset(standard_dataflow)}")

# Requête sur un dataset Comext (import/export par produit)
# La sélection du bon endpoint est transparente pour l'utilisateur
try:
    df_comext = client.get_data(
        dataflow="DS-045409",
        dimensions={
            "FLOW": "1",        # Importations
            "REPORTER": "FR",   # France
            "PRODUCT": "27",    # Produits énergétiques (section SH)
        },
        last_n_observations=4,
    )
    print(f"\nNombre de lignes récupérées (Comext): {len(df_comext)}")
    df_comext.head()
except Exception as e:
    print(f"\nErreur lors de la requête Comext: {type(e).__name__}: {str(e)[:120]}")

### 1.9 - Utilisation en tant que context manager <a id="section-1.9"></a>

`EurostatClient` implémente le protocole de context manager (`__enter__` / `__exit__`). Son utilisation avec `with` garantit la fermeture propre des connexions réseau, même en cas d'exception.

In [ ]:
# Utilisation recommandée : context manager
with EurostatClient() as eurostat:
    # Récupération de la structure
    structure = eurostat.get_structure("une_rt_m")
    print(f"Structure chargée pour '{structure.dataflow}' ({structure.num_dimensions} dimensions)")

    # Récupération des données
    df = eurostat.get_data(
        dataflow="une_rt_m",
        dimensions={
            "GEO": ["FR", "DE"],
            "FREQ": "M",
            "AGE": "TOTAL",
            "SEX": "T",
            "UNIT": "PC_ACT",
        },
        last_n_observations=12
    )

# Hors du bloc : les connexions sont fermées automatiquement
print(f"\nNombre de lignes récupérées: {len(df)}")
df.head()

In [ ]:
# Enregistrement manuel d'une structure pré-chargée
# Evite les appels d'API répétés lors d'un traitement en lot
client_batch = EurostatClient(auto_fetch_structure=False)

# Chargement explicite de la structure une seule fois
structure = client.get_structure("namq_10_gdp")
client_batch.register_structure(structure)

# Les appels suivants utilisent le cache sans appel réseau
df_cached = client_batch.get_data(
    dataflow="namq_10_gdp",
    dimensions={"GEO": "FR", "FREQ": "Q", "NA_ITEM": "B1GQ", "UNIT": "CP_MEUR"},
    last_n_observations=4,
)
print(f"Données récupérées via cache de structure: {len(df_cached)} lignes")

### 1.10 - Gestion des erreurs <a id="section-1.10"></a>

Démonstration de la gestion des erreurs avec des requêtes invalides.

In [ ]:
# Test 1 : Dataflow inexistant
print("Test 1: Dataflow inexistant")
try:
    df = client.get_data(
        dataflow="INVALID_DATAFLOW_XYZ",
        dimensions={"GEO": "FR"},
    )
except Exception as e:
    print(f"✓ Erreur attendue: {type(e).__name__}")
    print(f"  Message: {str(e)[:120]}")

# Test 2 : Dimension invalide (validation via structure)
print("\nTest 2: Nom de dimension invalide")
try:
    df = client.get_data(
        dataflow="namq_10_gdp",
        dimensions={"PAYS": ["FR"]},  # Devrait être GEO
    )
except Exception as e:
    print(f"✓ Erreur attendue: {type(e).__name__}")
    print(f"  Message: {str(e)[:120]}")

# Test 3 : Dépassement de max_split_combinations
print("\nTest 3: Dépassement de max_split_combinations")
try:
    df = client.get_data(
        dataflow="namq_10_gdp",
        dimensions={
            "GEO": ["FR", "DE", "IT", "ES", "NL", "BE", "PT", "AT", "PL", "SE"],
            "NA_ITEM": ["B1GQ", "P3", "P5G", "P6", "P7", "B11"],
        },
        split_dimensions=["GEO", "NA_ITEM"],
        max_split_combinations=50,  # 10 × 6 = 60 > 50
    )
except Exception as e:
    print(f"✓ Erreur attendue: {type(e).__name__}")
    print(f"  Message: {str(e)[:120]}")

print("\n✓ Tests de gestion d'erreurs terminés")

### 1.11 - Memento et conseils de performance <a id="section-1.11"></a>

Meilleures pratiques pour optimiser les téléchargements de données Eurostat.

### 1. Rate Limiting
- Le client charge automatiquement la configuration depuis `parameters/eurostat.json` si le fichier existe
- En l'absence de configuration, aucun rate limiting n'est appliqué
- Configuration recommandée : **30 requêtes par minute**

```python
from macroforecast.datasets.core.rate_limiter import RateLimiter

# Instanciation manuelle avec rate limiting explicite
client = EurostatClient(
    rate_limiter=RateLimiter(requests=30, unit="minutes", count=1)
)
```

### 2. Context Manager
- Toujours utiliser `with EurostatClient() as client:` en production
- Garantit la fermeture propre des connexions réseau

```python
with EurostatClient() as client:
    df = client.get_data(dataflow="namq_10_gdp", dimensions={"GEO": "FR"})
```

### 3. Split Dimensions
- Utiliser `split_dimensions` pour les requêtes avec de nombreuses valeurs par dimension
- Génère plusieurs petites requêtes plutôt qu'une seule requête volumineuse
- Exemple : 10 pays × 6 indicateurs = 60 requêtes avec `split_dimensions=["GEO", "NA_ITEM"]`

```python
df = client.get_data(
    dataflow="namq_10_gdp",
    dimensions={"GEO": ["FR", "DE", "IT", "ES", "NL", "BE"]},
    split_dimensions=["GEO"]
)
```

### 4. Cache de structure
- Utiliser `auto_fetch_structure=False` + `register_structure()` pour les traitements en lot
- Evite un appel réseau par requête pour récupérer la structure du dataflow

```python
client = EurostatClient(auto_fetch_structure=False)
structure = client.get_structure("namq_10_gdp")
client.register_structure(structure)

# Les appels suivants n'effectuent plus d'appel réseau pour la structure
for country in countries:
    df = client.get_data(dataflow="namq_10_gdp", dimensions={"GEO": country})
```

### 5. Datasets Comext
- Les datasets `DS-*` utilisent automatiquement l'API Comext
- Aucune configuration supplémentaire n'est requise

```python
df = client.get_data(
    dataflow="DS-045409",  # Bascule automatiquement vers l'API Comext
    dimensions={"FLOW": "1", "REPORTER": "FR"},
    last_n_observations=4,
)
```

### 6. Formats de réponse
- **CSV** (défaut) : format SDMX-CSV 3.0, le plus robuste pour pandas
- **JSON** : JSON-stat 2.0, utile pour les intégrations JavaScript
- **TSV** : format héritage, à éviter sauf contrainte externe
- **XML** : SDMX-ML 3.0, pour les usages avancés

```python
df = client.get_data(
    dataflow="namq_10_gdp",
    dimensions={"GEO": "FR"},
    format=EurostatResponseFormat.JSON  # Format alternatif
)
```

### 7. Filtrage temporel
- Préférer `last_n_observations` pour ne charger que les données récentes
- Utiliser `start_period` / `end_period` pour une fenêtre fixe
- Le format de période suit la norme SDMX : `"2024"`, `"2024-Q1"`, `"2024-01"`

```python
# Seulement les 4 derniers trimestres
df = client.get_data(
    dataflow="namq_10_gdp",
    dimensions={"GEO": "FR", "NA_ITEM": "B1GQ"},
    last_n_observations=4
)
```

### 8. Gestion des doublons
- Par défaut : `on_duplicate="warn"` (affiche un warning dans les logs)
- Options : `"ignore"`, `"warn"`, `"raise"`

```python
df = client.get_data(
    dataflow="namq_10_gdp",
    dimensions={"GEO": "FR"},
    on_duplicate="raise"  # Exception immédiate en cas de doublon
)
```

## Conclusion

Ce notebook a démontré toutes les fonctionnalités principales :

- du client Eurostat (énumération des dataflows, extraction de leur structure, exécution de requêtes avec différents paramètres, gestion des datasets Comext, utilisation en tant que context manager ...) ;

Pour plus d'informations, consulter :
- Le fichier `macroforecast/datasets/sources/eurostat.py`